# GeoMamba — FİNAL GRID — D2 genelleme + D1 faktöriyel (son kota) (Kaggle T4)

**Kurulum (bir kere):**
1. Sağdaki panel → **Session options** → Accelerator: **GPU T4**, Internet: **On**
2. Üst menü → **Add-ons → Secrets** → `GH_TOKEN` ekleyin.
   Token: github.com → Settings → Developer settings → Fine-grained tokens →
   sadece `geomamba` reposu, **Contents: Read-only**.
3. **Run All**. Grid ~2-3 saat sürer; sonuçlar `experiments/grid_results.csv`.
4. Bitince **Save Version** → çıktıdan `grid_results.csv`'yi indirin.

**ÖNEMLİ — kalıcılık:** Bu notebook'u interaktif Run All ile DEĞİL,
**Save Version → Save & Run All (Commit)** ile çalıştırın. İnteraktif
session kapanınca çıktılar silinir; Save & Run All çıktıyı kalıcı yapar.

**GPU:** Session options'ta **T4 x2** seçerseniz grid otomatik iki parçaya bölünüp iki GPU'da paralel koşar (süre yarıya iner).


In [ ]:
%cd /kaggle/working
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("GH_TOKEN")
!rm -rf /kaggle/working/geomamba
!git clone --depth 1 https://{token}@github.com/yusuftiryaki/geomamba.git /kaggle/working/geomamba


In [ ]:
%cd /kaggle/working/geomamba
import os, torch
os.environ["PYTHONPATH"] = "/kaggle/working/geomamba/src"
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "YOK - Session options'tan GPU secin!")


In [ ]:
import os, subprocess, sys, pathlib
GRID = "configs/grid/m4_final.yaml"

n_gpu = int(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.count("GPU "))
print("GPU sayisi:", n_gpu)

# datasets once, then one process per GPU
subprocess.run([sys.executable, "scripts/run_grid.py", "--grid", GRID, "--data-only"], check=True)
procs = []
for k in range(max(n_gpu, 1)):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(k))
    log = open(f"shard{k}.log", "w")
    procs.append((k, log, subprocess.Popen(
        [sys.executable, "scripts/run_grid.py", "--grid", GRID, "--skip-data",
         "--shard", f"{k}/{max(n_gpu, 1)}"],
        env=env, stdout=log, stderr=subprocess.STDOUT)))
print(f"{len(procs)} shard baslatildi; loglar shard*.log")
for k, log, p in procs:
    rc = p.wait(); log.close()
    print(f"shard {k} bitti, rc={rc}")
    print(pathlib.Path(f"shard{k}.log").read_text()[-2000:])


In [ ]:
import glob, shutil, pandas as pd
frames = [pd.read_csv(f) for f in sorted(glob.glob("experiments/grid_results*.csv"))]
if frames:
    df = pd.concat(frames, ignore_index=True)
    df.to_csv("/kaggle/working/grid_results.csv", index=False)
    for f in glob.glob("experiments/grid_results*.json"):
        shutil.copy(f, "/kaggle/working/")
    display(df)
else:
    print("sonuc yok - shard*.log dosyalarina bakin")
